# Rekurrente neuronale Netzwerke

Im vorherigen Modul haben wir umfangreiche semantische Repräsentationen von Text behandelt. Die Architektur, die wir verwendet haben, erfasst die aggregierte Bedeutung von Wörtern in einem Satz, berücksichtigt jedoch nicht die **Reihenfolge** der Wörter, da die Aggregationsoperation, die auf die Einbettungen folgt, diese Information aus dem ursprünglichen Text entfernt. Da diese Modelle die Wortreihenfolge nicht darstellen können, sind sie nicht in der Lage, komplexere oder mehrdeutige Aufgaben wie Textgenerierung oder Fragebeantwortung zu lösen.

Um die Bedeutung einer Textsequenz zu erfassen, verwenden wir eine neuronale Netzwerkarchitektur, die als **rekurrentes neuronales Netzwerk** oder RNN bezeichnet wird. Bei der Verwendung eines RNN führen wir unseren Satz Token für Token durch das Netzwerk, und das Netzwerk erzeugt einen **Zustand**, den wir dann zusammen mit dem nächsten Token erneut an das Netzwerk weitergeben.

![Bild zeigt ein Beispiel für eine rekurrente neuronale Netzwerkgenerierung.](../../../../../translated_images/de/rnn.27f5c29c53d727b5.webp)

Gegeben die Eingabesequenz von Tokens $X_0,\dots,X_n$, erstellt das RNN eine Sequenz von neuronalen Netzwerkblöcken und trainiert diese Sequenz Ende-zu-Ende mithilfe von Backpropagation. Jeder Netzwerkblock nimmt ein Paar $(X_i,S_i)$ als Eingabe und erzeugt $S_{i+1}$ als Ergebnis. Der Endzustand $S_n$ oder die Ausgabe $Y_n$ wird in einen linearen Klassifikator eingespeist, um das Ergebnis zu erzeugen. Alle Netzwerkblöcke teilen dieselben Gewichte und werden mit einem einzigen Backpropagation-Durchlauf Ende-zu-Ende trainiert.

> Die obige Abbildung zeigt das rekurrente neuronale Netzwerk in entrollter Form (links) und in kompakterer rekurrenter Darstellung (rechts). Es ist wichtig zu erkennen, dass alle RNN-Zellen dieselben **teilbaren Gewichte** haben.

Da die Zustandsvektoren $S_0,\dots,S_n$ durch das Netzwerk geleitet werden, kann das RNN sequentielle Abhängigkeiten zwischen Wörtern lernen. Zum Beispiel kann es, wenn das Wort *nicht* irgendwo in der Sequenz auftaucht, bestimmte Elemente innerhalb des Zustandsvektors negieren.

Jede RNN-Zelle enthält zwei Gewichtsmatrizen: $W_H$ und $W_I$, sowie einen Bias $b$. Bei jedem RNN-Schritt wird gegeben die Eingabe $X_i$ und den Eingabestatus $S_i$ der Ausgabestatus als $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$ berechnet, wobei $f$ eine Aktivierungsfunktion ist (oft $\tanh$).

> Für Aufgaben wie die Textgenerierung (die wir in der nächsten Einheit behandeln) oder die maschinelle Übersetzung möchten wir auch bei jedem RNN-Schritt einen Ausgabewert erhalten. In diesem Fall gibt es auch eine weitere Matrix $W_O$, und die Ausgabe wird berechnet als $Y_i=f(W_O\times S_i+b_O)$.

Sehen wir uns an, wie rekurrente neuronale Netzwerke uns dabei helfen können, unseren Nachrichten-Datensatz zu klassifizieren.

> Für die Sandbox-Umgebung müssen wir die folgende Zelle ausführen, um sicherzustellen, dass die erforderliche Bibliothek installiert und die Daten vorgeladen sind. Wenn Sie lokal ausführen, können Sie die folgende Zelle überspringen.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Beim Training großer Modelle kann die GPU-Speicherzuweisung zum Problem werden. Außerdem müssen wir möglicherweise mit verschiedenen Minibatch-Größen experimentieren, damit die Daten in unseren GPU-Speicher passen, das Training aber gleichzeitig schnell genug ist. Wenn Sie diesen Code auf Ihrer eigenen GPU-Maschine ausführen, können Sie mit der Anpassung der Minibatch-Größe experimentieren, um das Training zu beschleunigen.

> **Hinweis**: Bestimmte Versionen von NVidia-Treibern sind dafür bekannt, den Speicher nach dem Training des Modells nicht freizugeben. In diesem Notebook führen wir mehrere Beispiele aus, und das könnte dazu führen, dass der Speicher in bestimmten Umgebungen erschöpft ist, insbesondere wenn Sie im Rahmen desselben Notebooks eigene Experimente durchführen. Wenn Sie beim Starten des Modelltrainings auf seltsame Fehler stoßen, sollten Sie den Notebook-Kernel neu starten.


In [3]:
batch_size = 16
embed_size = 64

## Einfacher RNN-Klassifikator

Im Fall eines einfachen RNN ist jede rekurrente Einheit ein einfaches lineares Netzwerk, das einen Eingabevektor und einen Zustandsvektor erhält und einen neuen Zustandsvektor erzeugt. In Keras kann dies durch die `SimpleRNN`-Schicht dargestellt werden.

Obwohl wir dem RNN-Layer direkt one-hot codierte Tokens übergeben können, ist dies aufgrund ihrer hohen Dimensionalität keine gute Idee. Daher verwenden wir eine Embedding-Schicht, um die Dimensionalität der Wortvektoren zu reduzieren, gefolgt von einer RNN-Schicht und schließlich einem `Dense`-Klassifikator.

> **Hinweis**: In Fällen, in denen die Dimensionalität nicht so hoch ist, zum Beispiel bei der Tokenisierung auf Zeichenebene, kann es sinnvoll sein, one-hot codierte Tokens direkt in die RNN-Zelle zu übergeben.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Hinweis:** Wir verwenden hier der Einfachheit halber eine untrainierte Einbettungsschicht, aber für bessere Ergebnisse können wir eine vortrainierte Einbettungsschicht mit Word2Vec verwenden, wie in der vorherigen Einheit beschrieben. Es wäre eine gute Übung für Sie, diesen Code so anzupassen, dass er mit vortrainierten Einbettungen funktioniert.

Trainieren wir nun unser RNN. RNNs sind im Allgemeinen recht schwierig zu trainieren, da die RNN-Zellen entlang der Sequenzlänge entrollt werden und die daraus resultierende Anzahl der beim Backpropagation beteiligten Schichten ziemlich groß ist. Daher müssen wir eine kleinere Lernrate wählen und das Netzwerk mit einem größeren Datensatz trainieren, um gute Ergebnisse zu erzielen. Dies kann recht lange dauern, daher wird die Verwendung einer GPU bevorzugt.

Um die Sache zu beschleunigen, trainieren wir das RNN-Modell nur mit Nachrichtentiteln und lassen die Beschreibung weg. Sie können versuchen, mit der Beschreibung zu trainieren und sehen, ob Sie das Modell zum Lernen bringen.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Hinweis:** Die Genauigkeit ist hier wahrscheinlich geringer, da wir nur mit Nachrichtentiteln trainieren.


## Rückblick auf Sequenzen variabler Länge 

Denken Sie daran, dass die `TextVectorization`-Schicht Sequenzen variabler Länge in einem Minibatch automatisch mit Pad-Token auffüllt. Es stellt sich heraus, dass diese Token auch am Training teilnehmen und die Konvergenz des Modells erschweren können.

Es gibt mehrere Ansätze, um die Menge an Padding zu minimieren. Einer davon ist, den Datensatz nach Sequenzlänge neu zu ordnen und alle Sequenzen nach Größe zu gruppieren. Dies kann mit der Funktion `tf.data.experimental.bucket_by_sequence_length` durchgeführt werden (siehe [Dokumentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Ein weiterer Ansatz ist die Verwendung von **Maskierung**. In Keras unterstützen einige Schichten zusätzliche Eingaben, die anzeigen, welche Token beim Training berücksichtigt werden sollen. Um Maskierung in unser Modell einzubinden, können wir entweder eine separate `Masking`-Schicht ([Docs](https://keras.io/api/layers/core_layers/masking/)) einfügen oder den `mask_zero=True`-Parameter unserer `Embedding`-Schicht angeben.

> **Hinweis**: Dieses Training dauert etwa 5 Minuten, um eine Epoche über den gesamten Datensatz abzuschließen. Sie können das Training jederzeit abbrechen, wenn Ihnen die Geduld ausgeht. Eine weitere Möglichkeit besteht darin, die Menge der für das Training verwendeten Daten zu begrenzen, indem Sie eine `.take(...)`-Klausel nach den `ds_train`- und `ds_test`-Datensätzen hinzufügen.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Da wir jetzt Maskierung verwenden, können wir das Modell mit dem gesamten Datensatz aus Titeln und Beschreibungen trainieren.

> **Hinweis**: Ist Ihnen aufgefallen, dass wir einen Vektorisierer verwenden, der auf den Nachrichtentiteln trainiert wurde und nicht auf dem gesamten Text des Artikels? Potenziell könnten dadurch einige Tokens ignoriert werden, deshalb wäre es besser, den Vektorisierer neu zu trainieren. Es könnte jedoch nur einen sehr kleinen Effekt haben, daher verwenden wir der Einfachheit halber weiterhin den vorherigen vortrainierten Vektorisierer.


## LSTM: Langzeit-Kurzzeitgedächtnis

Eines der Hauptprobleme von RNNs sind **verschwindende Gradienten**. RNNs können ziemlich lang sein und haben möglicherweise Schwierigkeiten, die Gradienten während der Rückpropagation bis zur ersten Schicht des Netzwerks zurückzuleiten. Wenn dies passiert, kann das Netzwerk keine Beziehungen zwischen weit auseinanderliegenden Tokens erlernen. Eine Möglichkeit, dieses Problem zu vermeiden, besteht darin, **explizite Zustandsverwaltung** durch die Verwendung von **Gates** einzuführen. Die zwei häufigsten Architekturen, die Gates einführen, sind **langzeit-kurzzeitgedächtnis** (LSTM) und **gated relay unit** (GRU). Hier behandeln wir LSTMs.

![Bild, das eine Beispiel-Langzeit-Kurzzeitgedächtniszelle zeigt](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Ein LSTM-Netzwerk ist ähnlich wie ein RNN organisiert, jedoch werden zwei Zustände von Schicht zu Schicht weitergegeben: der tatsächliche Zustand $c$ und der verborgene Vektor $h$. In jeder Einheit wird der verborgene Vektor $h_{t-1}$ mit dem Input $x_t$ kombiniert, und zusammen steuern sie, was mit dem Zustand $c_t$ und dem Output $h_t$ durch **Gates** passiert. Jedes Gate hat eine Sigmoid-Aktivierung (Ausgabe im Bereich $[0,1]$), die als bitweise Maske betrachtet werden kann, wenn sie mit dem Zustandsvektor multipliziert wird. LSTMs haben die folgenden Gates (von links nach rechts auf dem obigen Bild):
* **Forget-Gate**, das bestimmt, welche Komponenten des Vektors $c_{t-1}$ wir vergessen müssen und welche weitergegeben werden.
* **Input-Gate**, das bestimmt, wie viel Information aus dem Eingangsvektor und dem vorherigen verborgenen Vektor in den Zustandsvektor eingearbeitet werden soll.
* **Output-Gate**, das den neuen Zustandsvektor nimmt und entscheidet, welche seiner Komponenten verwendet werden, um den neuen verborgenen Vektor $h_t$ zu erzeugen.

Die Komponenten des Zustands $c$ können als Flags betrachtet werden, die an- und ausgeschaltet werden können. Wenn wir beispielsweise den Namen *Alice* in der Sequenz treffen, vermuten wir, dass er sich auf eine Frau bezieht, und setzen das Flag im Zustand, das anzeigt, dass wir ein weibliches Substantiv im Satz haben. Wenn wir anschließend die Wörter *und Tom* antreffen, setzen wir das Flag, das anzeigt, dass wir ein Plural-Substantiv haben. Durch die Manipulation des Zustands können wir somit die grammatikalischen Eigenschaften des Satzes nachverfolgen.

> **Hinweis**: Hier ist eine großartige Ressource zum Verständnis der internen Abläufe von LSTMs: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) von Christopher Olah.

Obwohl die interne Struktur einer LSTM-Zelle komplex erscheinen kann, verbirgt Keras diese Implementierung in der `LSTM` Schicht, sodass wir im obigen Beispiel nur die rekurrente Schicht ersetzen müssen:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Beachten** Sie, dass das Training von LSTMs ebenfalls ziemlich langsam ist, und Sie anfangs möglicherweise keine große Steigerung der Genauigkeit sehen. Sie müssen möglicherweise eine Weile weiter trainieren, um eine gute Genauigkeit zu erreichen.


## Bidirektionale und mehrschichtige RNNs

In unseren bisherigen Beispielen arbeiten die rekurrenten Netzwerke vom Anfang einer Sequenz bis zum Ende. Das erscheint uns natürlich, da es der gleichen Richtung folgt, in der wir lesen oder Sprache hören. Für Szenarien, die einen Zufallszugriff auf die Eingabesequenz erfordern, ist es jedoch sinnvoller, die rekurrente Berechnung in beiden Richtungen durchzuführen. RNNs, die Berechnungen in beide Richtungen zulassen, nennt man **bidirektionale** RNNs, und sie können erstellt werden, indem man die rekurrente Schicht mit einer speziellen `Bidirectonal`-Schicht umhüllt.

> **Hinweis**: Die `Bidirectional`-Schicht macht zwei Kopien der Schicht innerhalb von ihr und setzt die `go_backwards`-Eigenschaft einer dieser Kopien auf `True`, sodass diese in die entgegengesetzte Richtung entlang der Sequenz läuft.

Rekurrente Netzwerke, ein- oder bidirektional, erfassen Muster innerhalb einer Sequenz und speichern sie in Zustandsvektoren oder geben sie als Ausgabe zurück. Wie bei Faltungsnetzwerken können wir eine weitere rekurrente Schicht nach der ersten hinzufügen, um höherstufige Muster zu erfassen, die aus niedrigstufigen Mustern bestehen, die von der ersten Schicht extrahiert wurden. Dies führt uns zum Konzept des **mehrschichtigen RNN**, das aus zwei oder mehreren rekurrenten Netzwerken besteht, wobei die Ausgabe der vorherigen Schicht als Eingabe an die nächste Schicht weitergegeben wird.

![Bild, das ein mehrschichtiges lang-kurzzeitgedächtnis-RNN zeigt](../../../../../translated_images/de/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Bild aus [diesem wunderbaren Beitrag](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) von Fernando López.*

Keras macht es einfach, diese Netzwerke zu konstruieren, weil man einfach mehr rekurrente Schichten zum Modell hinzufügen muss. Für alle Schichten außer der letzten müssen wir den `return_sequences=True`-Parameter angeben, weil wir von der Schicht alle Zwischenzustände zurückbekommen müssen, und nicht nur den Endzustand der rekurrenten Berechnung.

Lassen Sie uns ein zweischichtiges bidirektionales LSTM für unser Klassifikationsproblem bauen.

> **Hinweis**: Dieser Code braucht wieder ziemlich lange, um fertig zu werden, aber er liefert uns die höchste Genauigkeit, die wir bisher gesehen haben. Vielleicht lohnt es sich also zu warten und das Ergebnis abzuwarten.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNNs für andere Aufgaben

Bis jetzt haben wir uns darauf konzentriert, RNNs zur Klassifizierung von Textsequenzen zu verwenden. Aber sie können viele weitere Aufgaben bewältigen, wie Textgenerierung und maschinelle Übersetzung &mdash; diese Aufgaben werden wir in der nächsten Einheit behandeln.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
